# Face Mask Detection (MobileNetV2)
Binary classifier (Mask / No Mask) with transfer learning plus real-time webcam detection.

**Pipeline:** setup → data → model → train → plots → save → live detection.
For webcam use outside notebooks, prefer `python -m src.detect`.

In [ ]:
# Run from project root even if the notebook lives in notebooks/
import os
if Path().resolve().name == 'notebooks':
    os.chdir(Path().resolve().parent)
print('Working dir:', Path().resolve())

import datetime
import json
import os
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ---- Config (edit here; all paths are relative) ----
DATA_DIR = Path('data/train')
MODEL_PATH = Path('models/mask_detector_mobilenet.keras')
LABELS_PATH = Path('models/class_indices.json')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
VALIDATION_SPLIT = 0.2


## 1. Data
Expected layout:
```
data/train/with_mask/*.jpg
data/train/without_mask/*.jpg
```
Folder names become class names. `validation_split=0.2` carves validation out of the same folder.

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
)

train_gen = datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=42,
)
val_gen = datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=42,
)
print('Class indices:', train_gen.class_indices)
print("Note: prob < 0.5 ->", min(train_gen.class_indices, key=train_gen.class_indices.get))


## 2. Model
Frozen MobileNetV2 (ImageNet) + GlobalAveragePooling + Dropout(0.5) + Sigmoid. Freezing the base keeps training fast and avoids overfitting on small data.

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## 3. Train

In [ ]:
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)


## 4. Training curves

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend(); plt.title('Accuracy')
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend(); plt.title('Loss')
plt.tight_layout()
Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/training_curves.png')
plt.show()


## 5. Save model + label mapping
`class_indices.json` records which folder was class 0 vs 1, so detection can never invert Mask/No-Mask.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)
with open(LABELS_PATH, 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print(f'Model saved to {MODEL_PATH}')
print(f'Labels saved to {LABELS_PATH}: {train_gen.class_indices}')


## 6. Real-time detection
Uses Haar cascade for faces + the trained classifier per face. Saves one frame per no-mask event (throttled to 1 per 2 s), beeps, writes `outputs/no_mask_log.txt`, records `outputs/session_video.avi`, and writes `outputs/session_report.txt`.

> Run this cell where a webcam + display exist. For regular use: `python -m src.detect`.

In [ ]:
# ---- Notebook-safe detection settings ----
SOURCE = 0          # webcam; falls back to demo video if unavailable
MAX_FRAMES = 60     # keep the notebook demo short (q/Esc also stop it)
COOLDOWN = 2.0      # min seconds between saved no-mask captures

def preprocess_face(face):
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face = cv2.resize(face, IMG_SIZE)
    return np.expand_dims(face / 255.0, axis=0)

def beep():
    try:
        import winsound
        winsound.Beep(880, 400)
    except Exception:
        print('\a', end='', flush=True)

def prob_to_label(prob, class_indices):
    if class_indices:
        inv = {v: k for k, v in class_indices.items()}
        raw = inv.get(int(prob >= 0.5), '')
        k = raw.lower()
        if 'without' in k or 'no' in k.replace('without', 'no'):
            return 'No Mask', (0, 0, 255)
        return 'Mask', (0, 255, 0)
    return ('Mask', (0, 255, 0)) if prob < 0.5 else ('No Mask', (0, 0, 255))

model = tf.keras.models.load_model(MODEL_PATH)
with open(LABELS_PATH) as f:
    class_indices = json.load(f)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened() and Path('outputs/demo_input.mp4').exists():
    print('Webcam unavailable, using outputs/demo_input.mp4')
    cap = cv2.VideoCapture('outputs/demo_input.mp4')
if not cap.isOpened():
    raise SystemExit('No webcam and no outputs/demo_input.mp4 found.')

Path('outputs/no_mask_captures').mkdir(parents=True, exist_ok=True)
writer = None
mask_count = no_mask_count = 0
timestamps = []
last_save = 0.0
start_time = datetime.datetime.now()
display = True  # set False on first imshow failure (headless machines)
print("Detection running. Press 'q' or Esc to stop (or wait for MAX_FRAMES).")

try:
    for _ in range(MAX_FRAMES):
        ret, frame = cap.read()
        if not ret:
            print('End of stream.')
            break
        frame = cv2.resize(frame, (640, 480))
        if writer is None:
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            writer = cv2.VideoWriter('outputs/session_video.avi', fourcc, 20.0, (640, 480))
            if not writer.isOpened():
                writer = cv2.VideoWriter('outputs/session_video.avi', cv2.VideoWriter_fourcc(*'mp4v'), 20.0, (640, 480))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        for (x, y, w, h) in face_cascade.detectMultiScale(gray, 1.1, 4):
            crop = frame[y:y+h, x:x+w]
            if crop.size == 0:
                continue
            prob = float(model.predict(preprocess_face(crop), verbose=0)[0][0])
            label, color = prob_to_label(prob, class_indices)
            if label == 'Mask':
                mask_count += 1
            else:
                no_mask_count += 1
                if time.monotonic() - last_save >= COOLDOWN:
                    last_save = time.monotonic()
                    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
                    timestamps.append(ts)
                    cv2.imwrite(f'outputs/no_mask_captures/no_mask_{ts}.jpg', frame)
                    with open('outputs/no_mask_log.txt', 'a') as f:
                        f.write(f'No Mask Detected at {ts}\n')
                    beep()
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
            cv2.putText(frame, f'{label} {prob:.2f}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        total = mask_count + no_mask_count
        pm = mask_count/total*100 if total else 0
        pn = no_mask_count/total*100 if total else 0
        cv2.putText(frame, f'Mask: {pm:.1f}% No Mask: {pn:.1f}%', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        if display:
            try:
                cv2.imshow('Mask Detection', frame)
                if cv2.waitKey(1) & 0xFF in (ord('q'), 27):
                    break
            except cv2.error:
                display = False  # headless: keep recording without preview
                print('No display available, continuing headless.')
        writer.write(frame)
finally:
    cap.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()
    with open('outputs/session_report.txt', 'w') as f:
        f.write('Session Report\n' + '='*20 + '\n')
        f.write(f'Session started: {start_time}\nSession ended: {datetime.datetime.now()}\n')
        f.write(f'Mask detections: {mask_count}\nNo-mask detections: {no_mask_count}\n')
    print(f'Done. Mask={mask_count} No-mask={no_mask_count}, report + video saved under outputs/')
